In [38]:
import pandas as pd
import numpy as np
import sys
sys.path.append("../")
from src.sources.sheets.reader import GoogleSheetReader

In [39]:
gsheets = GoogleSheetReader()

In [40]:
google_sheet_info = {
    "sheet_name": "[MKP] Precios no duplicados",
    "worksheet": "price_data_mx"
}
df_inventory = gsheets.read_sheet(google_sheet_info)
df_inventory = df_inventory[["code", "brand", "model", "status"]]

In [41]:
df_inventory = df_inventory[df_inventory["status"].isin(["available", "no_stock"])].drop_duplicates(subset=["code"], keep="first").reset_index(drop=True)
df_inventory

,code,brand,model,status
0,MX1779-hero-hunk-150,Hero,Hunk 150,available
1,MX3016-vento-crossmax-300-rally-trx,Vento,Crossmax 300 Rally TRX,available
2,MX3144-cf-moto-uforce-600,CF Moto,UFORCE 600,available
3,MX2391-honda-crf50f,Honda,CRF50F,available
4,MX2862-vento-colt-300,Vento,Colt 300,available
...,...,...,...,...
697,MX2694-cf-moto-uforce-800-xl,CF Moto,UFORCE 800 XL,available
698,MX3339-veloci-argent-x3,Veloci,Argent X3,available
699,MX538-honda-xr-190,Honda,XR 190 L,no_stock
700,MX981-suzuki-gsxr-1000,Suzuki,GSXR-1000,available


In [42]:
SHEET_NAME = "Copy of  🌷👩‍👧💛 Lista de Precios Mayo | 2026"
marcas = ["Bajaj", "TVS", "Vento", "Yamaha", "Hero", "Honda", "Suzuki", "Italika", "Morbidelli", "CF Moto & CF LITE"]

In [43]:
hojas = gsheets.read_sheets_by_brands(SHEET_NAME, marcas)


[OK]   Bajaj → 'Bajajmanía V.3' (columna 'Desc. Bajaj' → 'Desc. marca')
[OK]   TVS → 'TVS' (columna 'Desc. TVS' → 'Desc. marca')
[OK]   Vento → 'Copy of VENTO | 15 al 14 Mayo' (columna 'Desc. Vento' → 'Desc. marca')
[OK]   Yamaha → 'YAMAHA' (columna 'Desc. Yamaha' → 'Desc. marca')
[OK]   Hero → 'HERO | 25 Mayo al 02 de Junio' (columna 'Desc. Hero' → 'Desc. marca')
[OK]   Honda → 'HONDA' (columna 'Desc. Honda' → 'Desc. marca')
[OK]   Suzuki → 'SUZUKI' (columna 'Desc. Suzuki' → 'Desc. marca')
[OK]   Italika → 'ITALIKA | 25 Mayo al 02 de Junio' (columna 'Desc. Italika' → 'Desc. marca')
[OK]   Morbidelli → 'MORBIDELLI | 25 Mayo al 02 de Junio' (columna 'Desc. Italika' → 'Desc. marca')
[OK]   CF Moto & CF LITE → 'CF MOTO & CF LITE' (columna 'Desc. CF' → 'Desc. marca')


In [44]:
df_all = pd.concat(hojas.values(), ignore_index=True)

In [26]:
df_columns_selected = df_all[["Marca", "Modelo MKP", "Año", "Desc. marca", "Desc. Galgo", "Total desc.", "Precio Galgo (c/IVA)"]]

In [27]:
df_columns_selected["has_galgo_discount"] = np.where(df_columns_selected["Desc. Galgo"] > 0, True, False)
df_columns_selected["has_brand_discount"] = np.where(df_columns_selected["Desc. marca"] > 0, True, False)
df_columns_selected["has_brand_and_galgo_discount"] = np.where(
    (df_columns_selected["has_galgo_discount"] > 0) & (df_columns_selected["has_brand_discount"] > 0),
    True,
    False
)

In [28]:
# df_columns_selected[df_columns_selected["has_brand_and_galgo_discount"] == True]
# df_columns_selected[df_columns_selected["has_galgo_discount"] == True]
# df_columns_selected[df_columns_selected["has_brand_discount"] == True]

In [29]:
df_columns_selected = df_columns_selected[df_columns_selected["Marca"].notna()]

In [30]:
df_columns_selected["Marca"] = df_columns_selected["Marca"].replace("GOES", "CF Moto")

In [31]:
df_columns_selected.rename(columns={"Marca": "brand", "Modelo MKP": "model", "Año": "year", "Desc. marca": "brand_discount", "Desc. Galgo": "galgo_discount", "Total desc.": "total_discount", "Precio Galgo (c/IVA)": "price_net"}, inplace=True)

In [32]:
df_merged = pd.merge(df_columns_selected, df_inventory[["code", "brand", "model"]], on = ["brand", "model"], how="left")

In [33]:
df_merged_columns_selected = df_merged[["code", "brand", "model", "year", "brand_discount", "galgo_discount", "total_discount", "price_net", "has_galgo_discount", "has_brand_discount", "has_brand_and_galgo_discount"]]

In [34]:
df_final = df_merged_columns_selected.sort_values(by="has_galgo_discount", ascending=False).reset_index(drop=True)

In [36]:
google_sheet_info = {
    "sheet_name": '[MKP - MX] Bonos en modelos',
    "worksheet": "mx",
    "df": df_final
}
gsheets.update_sheet(google_sheet_info, clear_data=True)
print(f"  ✓ Actualizado: Precios no duplicados")

Updated sheet: [MKP - MX] Bonos en modelos
  ✓ Actualizado: Precios no duplicados
